# rollback实验核心代码

本 Notebook 按照“发现 SGX Sealing 的数据新鲜性缺口 → 复现回滚攻击 → 实现可信版本见证 → 成功检测并拒绝回滚”的顺序整理。

代码都在`SocProject-Trusted-Password-Manager-built-on-Intel-SGX/sgx-vault-EN`下面。

## 核心结论

SGX Sealing 能验证密封文件的机密性和完整性，但无法单独判断一份合法密封文件是否为最新状态。本实验通过回放 V1 成功恢复旧密码，证明回滚攻击成立；随后引入独立可信见证端保存 V2，在再次回放 V1 时返回 `ROLLBACK`，并在密码读取前终止流程，成功抵御回滚攻击。

SGX本身的不足来自 **SGX Sealing 缺少独立的数据新鲜性保证**，而不是 SGX 的机密性或完整性机制失效。

## 核心文件与行号总览

| 作用 | 文件 | 行号 |
|---|---|---:|
| 密封状态中的版本字段 | `Enclave/Enclave.cpp` | 37–48 |
| 初始化状态版本 | `Enclave/Enclave.cpp` | 137–162 |
| 更新后递增版本 | `Enclave/Enclave.cpp` | 259–280 |
| 读取状态版本的 ECALL | `Enclave/Enclave.cpp` | 330–340 |
| SGX 解封逻辑及其新鲜性边界 | `Enclave/Enclave.cpp` | 391–422 |
| 无防护回滚攻击复现 | `bin/run_hw_attack.sh` | 17–28 |
| 可信版本保存与回滚判定 | `bin/witness_server.cpp` | 34–58 |
| 见证通信与退出码 | `bin/witness_client.cpp` | 24–30 |
| 有防护、fail-closed 实验路径 | `bin/run_hw_protected_client.sh` | 20–44 |

## 1. 在密封数据中加入状态版本

源文件：`Enclave/Enclave.cpp`，第 **37–48 行**。

`VaultWire` 是被 SGX Sealing 整体加密的密码库状态。`state_version` 用来描述业务状态的新旧，`vault_id` 用于让见证端区分不同密码库。版本号虽然受到密封完整性保护，但旧密封文件中的旧版本号本身仍然是合法的，因此还需要独立可信版本源。

```cpp
    struct VaultWire
    {
        uint32_t format_version;
        uint32_t credential_count;
        uint64_t state_version;
        uint8_t vault_id[16];
        /* 密码库级随机主密钥只存在于可信内存与 sealing 加密载荷中，不跨 ECALL 暴露。 */
        uint8_t master_key[32];
        sgx_sha256_hash_t pin_hash[PIN_HASH_SIZE];
        uint8_t pin_salt[PIN_SALT_SIZE];
        CredentialWire credentials[VAULT_MAX_CREDENTIALS];
    };
```

## 2. 初始化并在状态变化时递增版本

源文件：`Enclave/Enclave.cpp`，第 **137–162 行**。

创建密码库时将 `state_version` 初始化为 1，并随机生成 `vault_id`。新增、更新和删除凭据时都会递增版本；下方同时列出更新操作中的关键递增逻辑。

```cpp
int ecall_vault_create(const char *pin)
{
    if (g_initialized || g_unlocked)
        return VAULT_ERR_ALREADY_INITIALIZED;
    if (!valid_c_string(pin, VAULT_PIN_MAX, false))
        return VAULT_ERR_INVALID_PARAMETER;
    secure_zero(&g_vault, sizeof(g_vault));
    g_vault.format_version = VAULT_FORMAT_VERSION;
    g_vault.state_version = 1;
    if (sgx_read_rand(g_vault.vault_id, sizeof(g_vault.vault_id)) != SGX_SUCCESS ||
        sgx_read_rand(g_vault.master_key, sizeof(g_vault.master_key)) != SGX_SUCCESS)
    {
        secure_zero(&g_vault, sizeof(g_vault));
        return VAULT_ERR_INTERNAL;
    }
    if (sgx_read_rand(g_vault.pin_salt, sizeof(g_vault.pin_salt)) != SGX_SUCCESS)
    {
        secure_zero(&g_vault, sizeof(g_vault));
        return VAULT_ERR_INTERNAL;
    }
    hash_pin(g_vault.pin_salt, sizeof(g_vault.pin_salt),
             pin, strlen(pin),
             g_vault.pin_hash);
    g_initialized = true;
    g_unlocked = true;
    return VAULT_OK;
```

## 3. 更新密码后递增状态版本

源文件：`Enclave/Enclave.cpp`，第 **259–280 行**。

密码更新成功后执行 `++g_vault.state_version`。因此，实验中保存旧密码的 V1 与保存新密码的 V2 会具有不同且递增的版本号。

```cpp
int ecall_vault_update(const char *service, const char *username, const char *password)
{
    if (!g_initialized)
        return VAULT_ERR_NOT_INITIALIZED;
    if (!g_unlocked)
        return VAULT_ERR_LOCKED;
    if (!valid_c_string(service, VAULT_SERVICE_MAX, false) ||
        !valid_c_string(username, VAULT_USERNAME_MAX, true) ||
        !valid_c_string(password, VAULT_PASSWORD_MAX, true))
        return VAULT_ERR_INVALID_PARAMETER;
    const int index = find_service(service);
    if (index < 0)
        return VAULT_ERR_NOT_FOUND;
    if (g_vault.state_version == UINT64_MAX)
        return VAULT_ERR_INTERNAL;
    CredentialWire &c = g_vault.credentials[index];
    secure_zero(c.username, sizeof(c.username));
    secure_zero(c.password, sizeof(c.password));
    copy_string(c.username, sizeof(c.username), username);
    copy_string(c.password, sizeof(c.password), password);
    ++g_vault.state_version;
    return VAULT_OK;
```

## 4. 向非可信端提供版本读取接口

源文件：`Enclave/Enclave.cpp`，第 **330–340 行**。

该 ECALL 从已经解封并登录的 Enclave 状态中读取格式版本与状态版本。防护流程只先读取版本，不在可信见证检查通过前执行 Get、List 或 Update，从而保持 fail-closed。

```cpp
int ecall_vault_get_versions(uint32_t *format_version, uint64_t *state_version)
{
    if (!g_initialized)
        return VAULT_ERR_NOT_INITIALIZED;
    if (!g_unlocked)
        return VAULT_ERR_LOCKED;
    if (format_version == nullptr || state_version == nullptr)
        return VAULT_ERR_INVALID_PARAMETER;
    *format_version = g_vault.format_version;
    *state_version = g_vault.state_version;
    return VAULT_OK;
```

## 5. SGX 解封只验证合法性，无法识别合法旧状态

源文件：`Enclave/Enclave.cpp`，第 **391–422 行**。

`sgx_unseal_data` 能验证密封数据是否可被当前 Enclave 正确解封，随后代码检查结构和字段合法性。但是这里没有可信的“最新版本”可供比较，所以一个未被篡改的旧密封文件仍会被接受。这正是实验三首先发现并复现的 SGX Sealing 数据新鲜性缺口。

```cpp
int ecall_vault_unseal(const uint8_t *sealed, uint32_t sealed_size)
{
    if (sealed == nullptr || sealed_size < sizeof(sgx_sealed_data_t))
        return VAULT_ERR_INVALID_PARAMETER;
    const sgx_sealed_data_t *blob = reinterpret_cast<const sgx_sealed_data_t *>(sealed);
    const uint32_t encrypted_size = sgx_get_encrypt_txt_len(blob);
    const uint32_t aad_size = sgx_get_add_mac_txt_len(blob);
    if (encrypted_size != sizeof(VaultWire) || aad_size != 0 ||
        sgx_calc_sealed_data_size(aad_size, encrypted_size) != sealed_size)
        return VAULT_ERR_FORMAT;

    VaultWire candidate;
    secure_zero(&candidate, sizeof(candidate));
    uint32_t plaintext_size = sizeof(candidate);
    const sgx_status_t status = sgx_unseal_data(
        blob, nullptr, nullptr, reinterpret_cast<uint8_t *>(&candidate), &plaintext_size);
    if (status != SGX_SUCCESS)
    {
        secure_zero(&candidate, sizeof(candidate));
        return VAULT_ERR_UNSEAL;
    }
    if (plaintext_size != sizeof(candidate) || !validate_wire(candidate))
    {
        secure_zero(&candidate, sizeof(candidate));
        return VAULT_ERR_CORRUPT;
    }
    secure_zero(&g_vault, sizeof(g_vault));
    memcpy(&g_vault, &candidate, sizeof(g_vault));
    secure_zero(&candidate, sizeof(candidate));
    g_initialized = true;
    g_unlocked = false;
    return VAULT_OK;
```

## 6. 无防护路径：用 V1 覆盖 V2，复现回滚攻击

源文件：`bin/run_hw_attack.sh`，第 **17–28 行**。

脚本先保存旧状态 V1，再更新密码并保存 V2。攻击阶段把 V1 复制回当前的 `vault.sealed`。若重新加载后出现 `hw_password_v1`，便证明 SGX 接受了旧但合法的密封文件，即回滚攻击成功。

```bash
run_app 1 1234 1234 3 github alice hw_password_v1 8 9 0 | tee "$EXP/hw_v1.log"
cp vault.sealed "$EXP/hw_vault_v1.sealed"
run_app 2 1234 5 github alice hw_password_v2 8 9 0 | tee "$EXP/hw_v2.log"
cp vault.sealed "$EXP/hw_vault_v2.sealed"

cp "$EXP/hw_vault_v1.sealed" vault.sealed
run_app 2 1234 8 4 github 0 | tee "$EXP/hw_rollback.log"
if grep -q 'hw_password_v1' "$EXP/hw_rollback.log"; then
    echo '[PASS] Unprotected HW rollback succeeded: SGX accepted an old but valid sealed file.'
else
    echo '[FAIL] The old password was not observed.'; exit 1
fi
```

## 7. 可信见证端：保存最新版本并判定回滚

源文件：`bin/witness_server.cpp`，第 **34–58 行**。

见证服务器位于独立机器，不随机器 A 上的 `vault.sealed` 一起回退。`COMMIT` 单调更新可信版本；`CHECK` 将本地加载版本与可信版本比较。当 `version < trusted` 时返回 `ROLLBACK`，这条比较是防御机制的核心判定。

```cpp
static std::string handle(const std::string &line, const std::string &db_path) {
    std::istringstream in(line);
    std::string op, vault_id, extra;
    std::uint64_t version = 0;
    if (!(in >> op >> vault_id >> version) || (in >> extra) || vault_id.size() != 32)
        return "ERROR bad_request
";
    for (char c : vault_id)
        if (!std::isxdigit(static_cast<unsigned char>(c))) return "ERROR bad_vault_id
";

    auto db = load_db(db_path);
    const std::uint64_t trusted = db.count(vault_id) ? db[vault_id] : 0;
    if (op == "COMMIT") {
        if (version < trusted) return "ROLLBACK " + std::to_string(trusted) + "
";
        db[vault_id] = version;
        if (!save_db(db_path, db)) return "ERROR database_write
";
        return "OK " + std::to_string(version) + "
";
    }
    if (op == "CHECK") {
        if (trusted == 0) return "UNREGISTERED 0
";
        if (version < trusted) return "ROLLBACK " + std::to_string(trusted) + "
";
        if (version > trusted) return "UNCOMMITTED " + std::to_string(trusted) + "
";
        return "OK " + std::to_string(trusted) + "
";
    }
    return "ERROR bad_operation
";
}
```

## 8. 见证客户端：把 ROLLBACK 映射为专用退出码

源文件：`bin/witness_client.cpp`，第 **24–30 行**。

客户端把 `CHECK/COMMIT + vault_id + version` 发送给见证服务器。服务器返回 `ROLLBACK` 时，客户端以退出码 10 结束，便于实验脚本可靠地区分“检测到回滚”和普通通信错误。

```cpp
    if (fd < 0) { std::cerr << "cannot connect to witness
"; return 2; }
    const std::string request = std::string(argv[3]) + " " + argv[4] + " " + argv[5] + "
";
    if (write(fd, request.data(), request.size()) != static_cast<ssize_t>(request.size())) return 2;
    std::string response; char buf[128]; ssize_t n;
    while ((n = read(fd, buf, sizeof(buf))) > 0) response.append(buf, static_cast<std::size_t>(n));
    close(fd); std::cout << response;
    return response.rfind("OK ", 0) == 0 ? 0 : (response.rfind("ROLLBACK ", 0) == 0 ? 10 : 11);
```

## 9. 有防护路径：先检查版本，检测失败时拒绝读取

源文件：`bin/run_hw_protected_client.sh`，第 **20–44 行**。

脚本提取 V1、V2 并确认版本递增，将 V2 提交给机器 B；随后在机器 A 上回放 V1。加载旧文件后只读取版本并请求见证检查。在收到退出码 10 后立即报告成功并退出，整个过程中没有执行密码读取，从而实现 fail-closed 的回滚防御。

```bash
run_app 1 1234 1234 3 github alice protected_password_v1 8 9 0 | tee "$EXP/protected_v1.log"
cp vault.sealed "$EXP/protected_v1.sealed"
V1="$(state_of <"$EXP/protected_v1.log")"

run_app 2 1234 5 github alice protected_password_v2 8 9 0 | tee "$EXP/protected_v2.log"
cp vault.sealed "$EXP/protected_v2.sealed"
V2="$(state_of <"$EXP/protected_v2.log")"
[[ -n "$V1" && -n "$V2" && "$V2" -gt "$V1" ]] || { echo '[FATAL] Failed to extract increasing versions.'; exit 1; }

"$CLIENT" "$HOST" "$PORT" COMMIT "$VID" "$V2"
cp "$EXP/protected_v1.sealed" vault.sealed

# 先只解封并读取版本；在见证通过前绝不执行 Get/List/Update。
run_app 2 1234 8 0 | tee "$EXP/protected_rollback_check.log"
LOADED="$(state_of <"$EXP/protected_rollback_check.log")"
set +e
REPLY="$("$CLIENT" "$HOST" "$PORT" CHECK "$VID" "$LOADED")"; RC=$?
set -e
echo "$REPLY" | tee -a "$EXP/protected_rollback_check.log"
if [[ $RC -eq 10 ]]; then
    echo "[PASS] The protected HW path rejected the rollback: local version=$LOADED, trusted version=$V2."
    echo '[PASS] Fail-closed: no password read was performed.'
    exit 0
fi
echo '[FAIL] The witness did not reject the old version.'; exit 1
```